In [1]:
from google.colab import files
uploaded = files.upload()


Saving BBC News Train.csv to BBC News Train.csv


In [2]:
import pandas as pd

df = pd.read_csv("BBC News Train.csv")

print(df.head())
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())

   ArticleId                                               Text  Category
0       1833  worldcom ex-boss launches defence lawyers defe...  business
1        154  german business confidence slides german busin...  business
2       1101  bbc poll indicates economic gloom citizens in ...  business
3       1976  lifestyle  governs mobile choice  faster  bett...      tech
4        917  enron bosses in $168m payout eighteen former e...  business

Shape: (1490, 3)

Columns: ['ArticleId', 'Text', 'Category']


In [3]:
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nCategories:")
print(df['Category'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1490 entries, 0 to 1489
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ArticleId  1490 non-null   int64 
 1   Text       1490 non-null   object
 2   Category   1490 non-null   object
dtypes: int64(1), object(2)
memory usage: 35.1+ KB
None

Missing values:
ArticleId    0
Text         0
Category     0
dtype: int64

Categories:
Category
sport            346
business         336
politics         274
entertainment    273
tech             261
Name: count, dtype: int64


In [4]:
df = df.rename(columns={
    'Text': 'content',
    'Category': 'category'
})

print(df.head())

   ArticleId                                            content  category
0       1833  worldcom ex-boss launches defence lawyers defe...  business
1        154  german business confidence slides german busin...  business
2       1101  bbc poll indicates economic gloom citizens in ...  business
3       1976  lifestyle  governs mobile choice  faster  bett...      tech
4        917  enron bosses in $168m payout eighteen former e...  business


In [5]:
df.to_csv("newsbot_dataset.csv", index=False)

print("Dataset prepared!")

Dataset prepared!


In [6]:
!pip install spacy nltk scikit-learn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 77.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [7]:
import nltk

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [9]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = nltk.word_tokenize(text)

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return " ".join(tokens)

In [11]:
import nltk
nltk.download('punkt_tab', quiet=True)

df['clean_content'] = df['content'].apply(preprocess)

df[['content', 'clean_content']].head()

,content,clean_content
0,worldcom ex-boss launches defence lawyers defe...,worldcom exboss launch defence lawyer defendin...
1,german business confidence slides german busin...,german business confidence slide german busine...
2,bbc poll indicates economic gloom citizens in ...,bbc poll indicates economic gloom citizen majo...
3,lifestyle governs mobile choice faster bett...,lifestyle governs mobile choice faster better ...
4,enron bosses in $168m payout eighteen former e...,enron boss payout eighteen former enron direct...


In [12]:
df['clean_content'] = df['content'].apply(preprocess)

df[['content', 'clean_content']].head()

,content,clean_content
0,worldcom ex-boss launches defence lawyers defe...,worldcom exboss launch defence lawyer defendin...
1,german business confidence slides german busin...,german business confidence slide german busine...
2,bbc poll indicates economic gloom citizens in ...,bbc poll indicates economic gloom citizen majo...
3,lifestyle governs mobile choice faster bett...,lifestyle governs mobile choice faster better ...
4,enron bosses in $168m payout eighteen former e...,enron boss payout eighteen former enron direct...


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

X = df['clean_content']
y = df['category']

vectorizer = TfidfVectorizer(max_features=5000)

X_tfidf = vectorizer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train, y_train)

predictions = model.predict(X_test)

In [15]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, predictions))

print("\nClassification Report:")
print(classification_report(y_test, predictions))

Accuracy: 0.9731543624161074

Classification Report:
               precision    recall  f1-score   support

     business       0.97      0.97      0.97        75
entertainment       1.00      1.00      1.00        46
     politics       0.93      0.95      0.94        56
        sport       0.98      1.00      0.99        63
         tech       0.98      0.95      0.96        58

     accuracy                           0.97       298
    macro avg       0.97      0.97      0.97       298
 weighted avg       0.97      0.97      0.97       298



In [16]:
import spacy

nlp = spacy.load("en_core_web_sm")

sample = df['content'].iloc[0]

doc = nlp(sample)

for ent in doc.ents:
    print(ent.text, "-", ent.label_)

first - ORDINAL
cynthia cooper - PERSON
us - GPE
2002 - DATE
5.7bn - MONEY
new york - GPE
wednesday - DATE
arthur andersen - PERSON
early 2001 and - DATE
2002 - DATE
scott sullivan - PERSON
sullivan - PERSON
worldcom s accounting - ORG
2001 - DATE
85 years - DATE
2004 - DATE
mci - ORG
last week - DATE
mci - ORG
6.75bn - MONEY


In [17]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

df['sentiment_score'] = df['content'].apply(
    lambda x: sia.polarity_scores(x)['compound']
)

df[['category', 'sentiment_score']].head()

,category,sentiment_score
0,business,-0.9701
1,business,0.7623
2,business,-0.9318
3,tech,0.9554
4,business,-0.9486


In [18]:
def sentiment_label(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment'] = df['sentiment_score'].apply(sentiment_label)

print(df['sentiment'].value_counts())

sentiment
Positive    1052
Negative     427
Neutral       11
Name: count, dtype: int64


In [20]:
import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

sample_text = df['content'].iloc[0]

tokens = nltk.word_tokenize(sample_text)

pos_tags = nltk.pos_tag(tokens)

print(pos_tags[:30])

[('worldcom', 'JJ'), ('ex-boss', 'JJ'), ('launches', 'NNS'), ('defence', 'NN'), ('lawyers', 'NNS'), ('defending', 'VBG'), ('former', 'JJ'), ('worldcom', 'NN'), ('chief', 'NN'), ('bernie', 'NN'), ('ebbers', 'NNS'), ('against', 'IN'), ('a', 'DT'), ('battery', 'NN'), ('of', 'IN'), ('fraud', 'NN'), ('charges', 'NNS'), ('have', 'VBP'), ('called', 'VBN'), ('a', 'DT'), ('company', 'NN'), ('whistleblower', 'NN'), ('as', 'IN'), ('their', 'PRP$'), ('first', 'JJ'), ('witness', 'NN'), ('.', '.'), ('cynthia', 'VB'), ('cooper', 'JJ'), ('worldcom', 'NN')]


In [21]:
doc = nlp(df['content'].iloc[0])

for token in doc[:20]:
    print(token.text, token.dep_, token.head.text)

worldcom compound launches
ex compound launches
- nsubj launches
boss nsubj launches
launches ROOT launches
defence compound lawyers
lawyers dobj launches
defending acl lawyers
former amod ebbers
worldcom compound ebbers
chief compound ebbers
bernie compound ebbers
ebbers dobj defending
against prep defending
a det battery
battery pobj against
of prep battery
fraud compound charges
charges pobj of
have aux called


In [22]:
df.to_csv('newsbot_dataset.csv', index=False)

print("Dataset saved successfully.")

Dataset saved successfully.
